# 01 — Data Collection and Validation

This notebook constructs two separate datasets for the Nordic Wind Decision System:

1. **Main historical panel:** homogeneous D2 fixed-lead forecasts from every provider that supplies usable historical wind forecasts over the common period.
2. **Exact-run robustness panel:** complete model runs selected as available by the 11:00 Europe/Copenhagen D-1 decision time.

The provider registry deliberately distinguishes:

- **mandatory long-history core providers:** GFS, ICON-EU and ECMWF;
- **candidate providers:** DMI, ARPEGE, UKMO, GEM, JMA, CMA and BOM.

Candidate providers are selected solely from data availability and wind-variable coverage. A provider is not forced into the main panel when the Previous Runs archive contains only null wind fields. The exact-run dataset remains separate and may contain a broader source set in the recent period.

The main panel supports individual LightGBM experts, equal weighting, constrained weighting, dynamic weighting, stacking and pooled modelling.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import time
import warnings
from datetime import date, datetime, time as dt_time, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable, Sequence
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Configuration


In [ ]:
RUN_MODE = "AUDIT"  # "AUDIT" or "FULL"
RUN_EXACT_ROBUSTNESS = True

# Main D2 Previous Runs panel.
MAIN_FULL_START_DATE = "2024-07-05"
MAIN_AUDIT_START_DATE = "2024-07-05"
MAIN_AUDIT_END_DATE = "2024-07-12"  # exclusive

# Exact Single Runs panel.
EXACT_FULL_START_DATE = "2025-09-01"
EXACT_AUDIT_START_DATE = "2026-04-01"
EXACT_AUDIT_END_DATE = "2026-04-04"  # exclusive

SETTLEMENT_SAFETY_LAG_DAYS = 16
PRICE_AREAS = ["DK1", "DK2"]
LOCAL_TIMEZONE = "Europe/Copenhagen"
DECISION_LOCAL_HOUR = 11
RANDOM_SEED = 42

MIN_CORE_SOURCE_COVERAGE = 0.99
MIN_OPTIONAL_SOURCE_COVERAGE = 0.995
MIN_COMMON_PANEL_RETENTION = 0.98

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "project.json"
if CONFIG_PATH.exists():
    with CONFIG_PATH.open("r", encoding="utf-8") as file:
        PROJECT_CONFIG = json.load(file)
else:
    PROJECT_CONFIG = {}

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
AUDIT_DIR = PROJECT_ROOT / "outputs" / "audits"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

for directory in [
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    AUDIT_DIR,
    FIGURE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

if RUN_MODE not in {"AUDIT", "FULL"}:
    raise ValueError("RUN_MODE must be 'AUDIT' or 'FULL'.")

LOCAL_TZ = ZoneInfo(LOCAL_TIMEZONE)

if RUN_MODE == "AUDIT":
    MAIN_START_DATE = MAIN_AUDIT_START_DATE
    MAIN_END_DATE = MAIN_AUDIT_END_DATE
    EXACT_START_DATE = EXACT_AUDIT_START_DATE
    EXACT_END_DATE = EXACT_AUDIT_END_DATE
else:
    MAIN_START_DATE = MAIN_FULL_START_DATE
    settled_cutoff = (
        pd.Timestamp.now(tz=LOCAL_TIMEZONE).normalize()
        - pd.Timedelta(days=SETTLEMENT_SAFETY_LAG_DAYS)
    )
    MAIN_END_DATE = settled_cutoff.date().isoformat()
    EXACT_START_DATE = EXACT_FULL_START_DATE
    EXACT_END_DATE = MAIN_END_DATE

START_LOCAL = pd.Timestamp(MAIN_START_DATE).tz_localize(LOCAL_TZ)
END_LOCAL = pd.Timestamp(MAIN_END_DATE).tz_localize(LOCAL_TZ)
START_UTC = START_LOCAL.tz_convert("UTC")
END_UTC = END_LOCAL.tz_convert("UTC")

EXACT_START_LOCAL = pd.Timestamp(EXACT_START_DATE).tz_localize(LOCAL_TZ)
EXACT_END_LOCAL = pd.Timestamp(EXACT_END_DATE).tz_localize(LOCAL_TZ)
EXACT_START_UTC = EXACT_START_LOCAL.tz_convert("UTC")
EXACT_END_UTC = EXACT_END_LOCAL.tz_convert("UTC")

if END_UTC <= START_UTC:
    raise ValueError("The main end date must be after the start date.")

EXPECTED_HOURS = len(
    pd.date_range(
        START_UTC,
        END_UTC,
        freq="h",
        inclusive="left",
    )
)
EXPECTED_TARGET_ROWS = EXPECTED_HOURS * len(PRICE_AREAS)

print("Project root:", PROJECT_ROOT)
print("Run mode:", RUN_MODE)
print("Main local interval:", START_LOCAL, "to", END_LOCAL, "(exclusive)")
print("Exact local interval:", EXACT_START_LOCAL, "to", EXACT_END_LOCAL, "(exclusive)")
print("Expected main target rows:", EXPECTED_TARGET_ROWS)


## 2. Spatial weather grid and provider registry

The main dataset uses D2 forecasts. GFS, ICON-EU and ECMWF are mandatory because the audit confirms complete historical wind coverage. Other providers remain candidates and are retained only when their Previous Runs data contain a sufficiently complete wind channel.

Wind height is selected per provider:

- prefer 100 m wind when complete;
- otherwise use 10 m wind when complete;
- exclude the provider from the main wind-expert panel when neither channel is usable.

This is valid for source-specific experts because weighting combines their predictions, not identical raw feature vectors.


In [ ]:
WEATHER_POINTS = pd.DataFrame(
    [
        {"point_id": "dk1_west_jutland_north", "price_area": "DK1", "latitude": 56.70, "longitude": 8.20,  "environment": "coastal",  "spatial_weight": 1.0},
        {"point_id": "dk1_west_jutland_south", "price_area": "DK1", "latitude": 55.55, "longitude": 8.35,  "environment": "coastal",  "spatial_weight": 1.0},
        {"point_id": "dk1_north_jutland",       "price_area": "DK1", "latitude": 57.10, "longitude": 9.65,  "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk1_central_jutland",     "price_area": "DK1", "latitude": 56.20, "longitude": 9.20,  "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk1_south_jutland",       "price_area": "DK1", "latitude": 55.15, "longitude": 9.10,  "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk1_funen",               "price_area": "DK1", "latitude": 55.35, "longitude": 10.25, "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk1_north_sea_west",      "price_area": "DK1", "latitude": 55.80, "longitude": 6.80,  "environment": "offshore", "spatial_weight": 1.0},
        {"point_id": "dk1_north_sea_north",     "price_area": "DK1", "latitude": 56.60, "longitude": 7.30,  "environment": "offshore", "spatial_weight": 1.0},
        {"point_id": "dk1_kattegat",            "price_area": "DK1", "latitude": 56.60, "longitude": 11.20, "environment": "offshore", "spatial_weight": 1.0},
        {"point_id": "dk2_west_zealand",        "price_area": "DK2", "latitude": 55.45, "longitude": 11.25, "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk2_east_zealand",        "price_area": "DK2", "latitude": 55.65, "longitude": 12.35, "environment": "coastal",  "spatial_weight": 1.0},
        {"point_id": "dk2_south_zealand",       "price_area": "DK2", "latitude": 54.85, "longitude": 11.90, "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk2_lolland_falster",     "price_area": "DK2", "latitude": 54.75, "longitude": 11.25, "environment": "coastal",  "spatial_weight": 1.0},
        {"point_id": "dk2_bornholm",            "price_area": "DK2", "latitude": 55.13, "longitude": 14.92, "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk2_baltic_west",         "price_area": "DK2", "latitude": 54.70, "longitude": 12.50, "environment": "offshore", "spatial_weight": 1.0},
        {"point_id": "dk2_baltic_east",         "price_area": "DK2", "latitude": 55.00, "longitude": 13.50, "environment": "offshore", "spatial_weight": 1.0},
        {"point_id": "dk2_oresund",             "price_area": "DK2", "latitude": 55.75, "longitude": 12.80, "environment": "offshore", "spatial_weight": 1.0},
    ]
)

MAIN_WEATHER_MODELS = {
    "gfs": {
        "api_model": "gfs_global",
        "required": True,
        "provider": "NOAA",
    },
    "icon_eu": {
        "api_model": "icon_eu",
        "required": True,
        "provider": "DWD",
    },
    "ecmwf": {
        "api_model": "ecmwf_ifs025",
        "required": True,
        "provider": "ECMWF",
    },
    "dmi": {
        "api_model": "dmi_harmonie_arome_europe",
        "required": False,
        "provider": "DMI",
    },
    "arpege_europe": {
        "api_model": "meteofrance_arpege_europe",
        "required": False,
        "provider": "Météo-France",
    },
    "ukmo_global": {
        "api_model": "ukmo_global_deterministic_10km",
        "required": False,
        "provider": "UK Met Office",
    },
    "gem_global": {
        "api_model": "gem_global",
        "required": False,
        "provider": "Environment Canada",
    },
    "jma_gsm": {
        "api_model": "jma_gsm",
        "required": False,
        "provider": "JMA",
    },
    "cma_grapes": {
        "api_model": "cma_grapes_global",
        "required": False,
        "provider": "CMA",
    },
    "bom_access": {
        "api_model": "bom_access_global",
        "required": False,
        "provider": "BOM",
    },
}

EXACT_WEATHER_MODELS = {
    "gfs": {
        "api_model": "gfs_global",
        "run_hours_utc": [0, 6, 12, 18],
        "availability_lag_hours": 6,
    },
    "icon_eu": {
        "api_model": "icon_eu",
        "run_hours_utc": [0, 6, 12, 18],
        "availability_lag_hours": 4,
    },
    "ecmwf": {
        "api_model": "ecmwf_ifs025",
        "run_hours_utc": [0, 6, 12, 18],
        "availability_lag_hours": 6,
    },
    "dmi": {
        "api_model": "dmi_harmonie_arome_europe",
        "run_hours_utc": list(range(0, 24, 3)),
        "availability_lag_hours": 4,
    },
    "arpege_europe": {
        "api_model": "meteofrance_arpege_europe",
        "run_hours_utc": [0, 6, 12, 18],
        "availability_lag_hours": 5,
    },
    "ukmo_global": {
        "api_model": "ukmo_global_deterministic_10km",
        "run_hours_utc": [0, 6, 12, 18],
        "availability_lag_hours": 6,
    },
    "gem_global": {
        "api_model": "gem_global",
        "run_hours_utc": [0, 12],
        "availability_lag_hours": 6,
    },
    "jma_gsm": {
        "api_model": "jma_gsm",
        "run_hours_utc": [0, 6, 12, 18],
        "availability_lag_hours": 6,
    },
    "cma_grapes": {
        "api_model": "cma_grapes_global",
        "run_hours_utc": [0, 6, 12, 18],
        "availability_lag_hours": 6,
    },
    "bom_access": {
        "api_model": "bom_access_global",
        "run_hours_utc": [0, 6, 12, 18],
        "availability_lag_hours": 6,
    },
}

WEATHER_VARIABLES = [
    "wind_speed_100m",
    "wind_direction_100m",
    "wind_speed_10m",
    "wind_direction_10m",
    "temperature_2m",
    "pressure_msl",
]

PREVIOUS_RUN_DAY = 2
PREVIOUS_RUN_VARIABLES = [
    f"{variable}_previous_day{PREVIOUS_RUN_DAY}"
    for variable in WEATHER_VARIABLES
]

display(WEATHER_POINTS)
display(pd.DataFrame(MAIN_WEATHER_MODELS).T)


## 3. HTTP, caching and provenance utilities


In [ ]:
class DataDownloadError(RuntimeError):
    pass


def stable_hash(value: Any) -> str:
    encoded = json.dumps(
        value,
        sort_keys=True,
        default=str,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()[:16]


def request_json(
    url: str,
    params: dict[str, Any] | None = None,
    timeout: int = 180,
    max_retries: int = 6,
    pause_seconds: float = 1.0,
) -> dict[str, Any] | list[Any]:
    last_error: Exception | None = None
    last_status: int | None = None
    last_body = ""

    for attempt in range(max_retries):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=timeout,
                headers={
                    "User-Agent": "nordic-wind-decision-system/0.2"
                },
            )
            last_status = response.status_code
            last_body = response.text[:1000]

            if response.status_code == 429:
                retry_after = float(
                    response.headers.get(
                        "Retry-After",
                        pause_seconds * (attempt + 1),
                    )
                )
                time.sleep(max(retry_after, pause_seconds))
                continue

            response.raise_for_status()
            return response.json()

        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt == max_retries - 1:
                break
            time.sleep(pause_seconds * (2 ** attempt))

    raise DataDownloadError(
        f"Request failed after {max_retries} attempts: {url}; "
        f"status={last_status}; body={last_body}"
    ) from last_error


def request_json_cached(
    url: str,
    params: dict[str, Any],
    cache_path: Path,
    force_refresh: bool = False,
) -> dict[str, Any] | list[Any]:
    if cache_path.exists() and not force_refresh:
        with cache_path.open("r", encoding="utf-8") as file:
            cached = json.load(file)
        return cached["payload"]

    payload = request_json(url=url, params=params)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    record = {
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "request_url": url,
        "request_params": params,
        "payload": payload,
    }

    with cache_path.open("w", encoding="utf-8") as file:
        json.dump(record, file, ensure_ascii=False)

    return payload


def trim_utc(
    frame: pd.DataFrame,
    timestamp_column: str,
    start_utc: pd.Timestamp = START_UTC,
    end_utc: pd.Timestamp = END_UTC,
) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()

    output = frame.copy()
    output[timestamp_column] = pd.to_datetime(
        output[timestamp_column],
        utc=True,
        errors="coerce",
    )

    return output.loc[
        output[timestamp_column].ge(start_utc)
        & output[timestamp_column].lt(end_utc)
    ].copy()


## 4. Energinet source registry and live metadata audit

The notebook records current source metadata before downloading observations. This makes dataset replacements and resolution changes visible in the Git-tracked audit outputs.


In [ ]:
EDS_BASE = "https://api.energidataservice.dk"

SOURCE_REGISTRY = pd.DataFrame(
    [
        {
            "dataset": "ProductionConsumptionSettlement",
            "role": "settlement target",
            "required": True,
            "information_time_use": "target only",
        },
        {
            "dataset": "DayAheadPrices",
            "role": "current day-ahead price",
            "required": True,
            "information_time_use": "market outcome/context",
        },
        {
            "dataset": "Elspotprices",
            "role": "historical day-ahead price fallback",
            "required": False,
            "information_time_use": "market outcome/context",
        },
        {
            "dataset": "ImbalancePrice",
            "role": "current imbalance price",
            "required": True,
            "information_time_use": "commercial outcome",
        },
        {
            "dataset": "RegulatingBalancePowerdata",
            "role": "historical imbalance fallback",
            "required": False,
            "information_time_use": "commercial outcome",
        },
        {
            "dataset": "Forecasts_Hour",
            "role": "Energinet operational wind forecast benchmark",
            "required": False,
            "information_time_use": "benchmark; revision timing to audit",
        },
    ]
)


def fetch_eds_metadata(dataset: str) -> dict[str, Any]:
    payload = request_json(f"{EDS_BASE}/meta/Dataset/{dataset}")
    if not isinstance(payload, dict):
        raise DataDownloadError(
            f"Unexpected metadata response for {dataset}."
        )
    return payload


metadata_rows: list[dict[str, Any]] = []
metadata_full: dict[str, Any] = {}

for dataset in SOURCE_REGISTRY["dataset"]:
    try:
        metadata = fetch_eds_metadata(dataset)
        metadata_full[dataset] = metadata
        metadata_rows.append(
            {
                "dataset": dataset,
                "title": metadata.get("title"),
                "resolution": metadata.get("resolution"),
                "data_from": metadata.get("dataFrom"),
                "active": metadata.get("active"),
                "last_metadata_update": metadata.get(
                    "lastMetadataUpdate"
                ),
                "metadata_status": "OK",
            }
        )
    except Exception as exc:
        metadata_rows.append(
            {
                "dataset": dataset,
                "metadata_status": f"ERROR: {exc!r}",
            }
        )

source_metadata_audit = SOURCE_REGISTRY.merge(
    pd.DataFrame(metadata_rows),
    on="dataset",
    how="left",
)

with (AUDIT_DIR / "energinet_source_metadata.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(metadata_full, file, indent=2, ensure_ascii=False)

source_metadata_audit.to_csv(
    AUDIT_DIR / "energinet_source_metadata_summary.csv",
    index=False,
)

display(source_metadata_audit)


## 5. Energinet download utilities


In [ ]:
def parse_eds_records(
    payload: dict[str, Any] | list[Any],
) -> list[dict[str, Any]]:
    if isinstance(payload, dict):
        for key in ["records", "result", "data"]:
            value = payload.get(key)
            if isinstance(value, list):
                return value

    if isinstance(payload, list):
        return payload

    raise DataDownloadError(
        f"Unexpected Energinet payload structure: {type(payload)}"
    )


def period_chunks(
    start: pd.Timestamp,
    end: pd.Timestamp,
    days: int,
) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    chunks: list[tuple[pd.Timestamp, pd.Timestamp]] = []
    cursor = start

    while cursor < end:
        right = min(cursor + pd.Timedelta(days=days), end)
        chunks.append((cursor, right))
        cursor = right

    return chunks


def format_eds_timestamp(
    timestamp: pd.Timestamp,
) -> str:
    timestamp = pd.Timestamp(timestamp)

    if timestamp.tzinfo is None:
        raise ValueError(
            "Energinet request timestamps must be timezone-aware."
        )

    return (
        timestamp
        .tz_convert("UTC")
        .strftime("%Y-%m-%dT%H:%M")
    )


def eds_download_chunked(
    dataset: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    filters: dict[str, list[str]] | None = None,
    chunk_days: int = 31,
    force_refresh: bool = False,
) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    chunks = period_chunks(start, end, chunk_days)

    for chunk_number, (left, right) in enumerate(
        chunks,
        start=1,
    ):
        params: dict[str, Any] = {
            "start": format_eds_timestamp(left),
            "end": format_eds_timestamp(right),
            "timezone": "UTC",
            "limit": 0,
        }

        if filters:
            params["filter"] = json.dumps(
                filters,
                separators=(",", ":"),
            )

        cache_key = stable_hash(
            {
                "dataset": dataset,
                "start": params["start"],
                "end": params["end"],
                "timezone": params["timezone"],
                "filters": filters,
            }
        )

        cache_path = (
            RAW_DIR
            / "energinet"
            / dataset
            / f"{cache_key}.json"
        )

        cache_existed_before_request = cache_path.exists()

        print(
            f"[{dataset}] chunk "
            f"{chunk_number}/{len(chunks)}: "
            f"{params['start']} to {params['end']} UTC"
        )

        payload = request_json_cached(
            url=f"{EDS_BASE}/dataset/{dataset}",
            params=params,
            cache_path=cache_path,
            force_refresh=force_refresh,
        )

        records = parse_eds_records(payload)

        if records:
            part = pd.DataFrame.from_records(records)
            part["_source_dataset"] = dataset
            part["_retrieved_from_cache"] = (
                cache_existed_before_request
                and not force_refresh
            )
            parts.append(part)

        time.sleep(0.2)

    if not parts:
        return pd.DataFrame()

    return pd.concat(parts, ignore_index=True)


def first_existing(
    columns: Iterable[str],
    candidates: Sequence[str],
) -> str:
    available = set(columns)

    for candidate in candidates:
        if candidate in available:
            return candidate

    raise KeyError(
        "None of the candidate columns exist: "
        f"{list(candidates)}"
    )


def numeric_series(
    frame: pd.DataFrame,
    column: str,
) -> pd.Series:
    return pd.to_numeric(
        frame[column],
        errors="coerce",
    )


## 6. Settled DK1/DK2 wind generation

Settlement data are used as the **target only**. They are not treated as if they were available in real time.


In [ ]:
def resolve_wind_component(
    frame: pd.DataFrame,
    aggregate_candidates: list[str],
    detailed_candidate_sets: list[list[str]],
    component_name: str,
) -> tuple[pd.Series, list[str]]:
    for column in aggregate_candidates:
        if column in frame.columns:
            return numeric_series(frame, column), [column]

    for column_set in detailed_candidate_sets:
        if all(column in frame.columns for column in column_set):
            values = pd.concat(
                [
                    numeric_series(frame, column)
                    for column in column_set
                ],
                axis=1,
            ).sum(axis=1, min_count=1)
            return values, column_set

    raise KeyError(
        f"Could not construct {component_name}. "
        f"Available columns: {frame.columns.tolist()}"
    )


def prepare_actual_wind(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    if frame.empty:
        raise ValueError(
            "ProductionConsumptionSettlement returned no records."
        )

    output = frame.copy()

    time_column = first_existing(
        output.columns,
        ["HourUTC", "TimeUTC", "QuarterUTC", "Minutes5UTC"],
    )
    area_column = first_existing(
        output.columns,
        ["PriceArea", "Area"],
    )

    output["timestamp_utc"] = pd.to_datetime(
        output[time_column],
        utc=True,
        errors="coerce",
    )
    output["price_area"] = output[area_column].astype(str)

    onshore, onshore_sources = resolve_wind_component(
        output,
        aggregate_candidates=[
            "OnshoreWindPower",
            "OnshoreWindPowerMWh",
        ],
        detailed_candidate_sets=[
            [
                "OnshoreWindGe50kW_MWh",
                "OnshoreWindLt50kW_MWh",
            ],
            [
                "OnshoreWindGe50kW",
                "OnshoreWindLt50kW",
            ],
        ],
        component_name="onshore wind",
    )

    offshore, offshore_sources = resolve_wind_component(
        output,
        aggregate_candidates=[
            "OffshoreWindPower",
            "OffshoreWindPowerMWh",
        ],
        detailed_candidate_sets=[
            [
                "OffshoreWindGe100MW_MWh",
                "OffshoreWindLt100MW_MWh",
            ],
            [
                "OffshoreWindGe100MW",
                "OffshoreWindLt100MW",
            ],
        ],
        component_name="offshore wind",
    )

    output["actual_onshore_wind_mwh"] = onshore
    output["actual_offshore_wind_mwh"] = offshore
    output["actual_wind_mwh"] = onshore + offshore
    output["wind_source_columns"] = (
        "onshore="
        + ",".join(onshore_sources)
        + "; offshore="
        + ",".join(offshore_sources)
    )

    result = output[
        [
            "timestamp_utc",
            "price_area",
            "actual_onshore_wind_mwh",
            "actual_offshore_wind_mwh",
            "actual_wind_mwh",
            "wind_source_columns",
        ]
    ].copy()

    result = trim_utc(result, "timestamp_utc")

    return (
        result.sort_values(["timestamp_utc", "price_area"])
        .drop_duplicates(
            ["timestamp_utc", "price_area"],
            keep="last",
        )
        .reset_index(drop=True)
    )


production_raw = eds_download_chunked(
    dataset="ProductionConsumptionSettlement",
    start=START_UTC - pd.Timedelta(days=1),
    end=END_UTC + pd.Timedelta(days=1),
    filters={"PriceArea": PRICE_AREAS},
    chunk_days=7 if RUN_MODE == "AUDIT" else 31,
)

actual_wind = prepare_actual_wind(production_raw)

print("Actual wind shape:", actual_wind.shape)
display(actual_wind.head())
display(
    actual_wind.groupby("price_area")[
        "actual_wind_mwh"
    ].describe()
)


## 7. Day-ahead prices

Both the current and historical Energinet datasets are attempted. Overlapping records prefer `DayAheadPrices`.


In [ ]:
PRICE_TIME_CANDIDATES = [
    "TimeUTC",
    "HourUTC",
    "QuarterUTC",
    "Minutes15UTC",
    "Minutes5UTC",
]

PRICE_VALUE_CANDIDATES = [
    "DayAheadPriceEUR",
    "SpotPriceEUR",
    "PriceEUR",
    "DayAheadPrice",
    "SpotPrice",
]


def prepare_day_ahead_prices(
    frame: pd.DataFrame,
    source_dataset: str,
) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()

    time_column = first_existing(
        frame.columns,
        PRICE_TIME_CANDIDATES,
    )
    area_column = first_existing(
        frame.columns,
        ["PriceArea", "Area"],
    )
    price_column = first_existing(
        frame.columns,
        PRICE_VALUE_CANDIDATES,
    )

    output = frame.copy()
    output["raw_timestamp_utc"] = pd.to_datetime(
        output[time_column],
        utc=True,
        errors="coerce",
    )
    output["timestamp_utc"] = (
        output["raw_timestamp_utc"].dt.floor("h")
    )
    output["price_area"] = output[area_column].astype(str)
    output["raw_price_eur_mwh"] = pd.to_numeric(
        output[price_column],
        errors="coerce",
    )
    output["price_source_dataset"] = source_dataset

    hourly = (
        output.groupby(
            [
                "timestamp_utc",
                "price_area",
                "price_source_dataset",
            ],
            as_index=False,
        )
        .agg(
            day_ahead_price_eur_mwh=(
                "raw_price_eur_mwh",
                "mean",
            ),
            day_ahead_intervals_per_hour=(
                "raw_timestamp_utc",
                "nunique",
            ),
        )
    )

    return trim_utc(hourly, "timestamp_utc")


price_parts: list[pd.DataFrame] = []
price_errors: list[dict[str, str]] = []

for dataset in ["Elspotprices", "DayAheadPrices"]:
    try:
        raw = eds_download_chunked(
            dataset=dataset,
            start=START_UTC - pd.Timedelta(days=1),
            end=END_UTC + pd.Timedelta(days=1),
            filters={"PriceArea": PRICE_AREAS},
            chunk_days=7 if RUN_MODE == "AUDIT" else 31,
        )
        prepared = prepare_day_ahead_prices(raw, dataset)
        if not prepared.empty:
            price_parts.append(prepared)
    except Exception as exc:
        price_errors.append(
            {"dataset": dataset, "error": repr(exc)}
        )

if not price_parts:
    raise ValueError(
        "No usable day-ahead price records were downloaded."
    )

day_ahead_prices = pd.concat(
    price_parts,
    ignore_index=True,
)

source_priority = {
    "Elspotprices": 1,
    "DayAheadPrices": 2,
}

day_ahead_prices["_source_priority"] = (
    day_ahead_prices["price_source_dataset"]
    .map(source_priority)
    .fillna(0)
)

day_ahead_prices = (
    day_ahead_prices.sort_values(
        [
            "timestamp_utc",
            "price_area",
            "_source_priority",
        ]
    )
    .drop_duplicates(
        ["timestamp_utc", "price_area"],
        keep="last",
    )
    .drop(columns="_source_priority")
    .reset_index(drop=True)
)

print("Day-ahead price shape:", day_ahead_prices.shape)
display(day_ahead_prices.head())
display(pd.DataFrame(price_errors))


## 8. Imbalance prices

The native-resolution table is preserved. A separate hourly mean is produced only for the first hourly modelling panel.


In [ ]:
IMBALANCE_TIME_CANDIDATES = [
    "TimeUTC",
    "QuarterUTC",
    "HourUTC",
    "Minutes15UTC",
    "Minutes5UTC",
]

IMBALANCE_VALUE_CANDIDATES = [
    "ImbalancePriceEUR",
    "ImbalancePrice",
    "BalancePowerPriceEUR",
    "BalancingPowerPriceEUR",
]


def prepare_imbalance_prices(
    frame: pd.DataFrame,
    source_dataset: str,
) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()

    time_column = first_existing(
        frame.columns,
        IMBALANCE_TIME_CANDIDATES,
    )
    area_column = first_existing(
        frame.columns,
        ["PriceArea", "Area"],
    )
    value_column = first_existing(
        frame.columns,
        IMBALANCE_VALUE_CANDIDATES,
    )

    output = frame.copy()
    output["timestamp_utc"] = pd.to_datetime(
        output[time_column],
        utc=True,
        errors="coerce",
    )
    output["price_area"] = output[area_column].astype(str)
    output["imbalance_price_eur_mwh"] = pd.to_numeric(
        output[value_column],
        errors="coerce",
    )
    output["imbalance_source_dataset"] = source_dataset

    keep_columns = [
        "timestamp_utc",
        "price_area",
        "imbalance_price_eur_mwh",
        "imbalance_source_dataset",
    ]

    return output[keep_columns].dropna(
        subset=["timestamp_utc"]
    )


imbalance_parts: list[pd.DataFrame] = []
imbalance_errors: list[dict[str, str]] = []

for dataset in [
    "RegulatingBalancePowerdata",
    "ImbalancePrice",
]:
    try:
        raw = eds_download_chunked(
            dataset=dataset,
            start=START_UTC - pd.Timedelta(days=1),
            end=END_UTC + pd.Timedelta(days=1),
            filters={"PriceArea": PRICE_AREAS},
            chunk_days=7 if RUN_MODE == "AUDIT" else 31,
        )
        prepared = prepare_imbalance_prices(raw, dataset)
        if not prepared.empty:
            imbalance_parts.append(prepared)
    except Exception as exc:
        imbalance_errors.append(
            {"dataset": dataset, "error": repr(exc)}
        )

if imbalance_parts:
    imbalance_native = pd.concat(
        imbalance_parts,
        ignore_index=True,
    )
    imbalance_native = trim_utc(
        imbalance_native,
        "timestamp_utc",
    )

    imbalance_priority = {
        "RegulatingBalancePowerdata": 1,
        "ImbalancePrice": 2,
    }
    imbalance_native["_source_priority"] = (
        imbalance_native["imbalance_source_dataset"]
        .map(imbalance_priority)
        .fillna(0)
    )
    imbalance_native = (
        imbalance_native.sort_values(
            [
                "timestamp_utc",
                "price_area",
                "_source_priority",
            ]
        )
        .drop_duplicates(
            ["timestamp_utc", "price_area"],
            keep="last",
        )
        .drop(columns="_source_priority")
        .reset_index(drop=True)
    )

    imbalance_hourly = imbalance_native.copy()
    imbalance_hourly["hour_utc"] = (
        imbalance_hourly["timestamp_utc"].dt.floor("h")
    )
    imbalance_hourly = (
        imbalance_hourly.groupby(
            ["hour_utc", "price_area"],
            as_index=False,
        )
        .agg(
            imbalance_price_eur_mwh=(
                "imbalance_price_eur_mwh",
                "mean",
            ),
            imbalance_intervals_per_hour=(
                "timestamp_utc",
                "nunique",
            ),
            imbalance_source_dataset=(
                "imbalance_source_dataset",
                "last",
            ),
        )
        .rename(columns={"hour_utc": "timestamp_utc"})
    )
else:
    imbalance_native = pd.DataFrame()
    imbalance_hourly = pd.DataFrame()

print("Native imbalance shape:", imbalance_native.shape)
print("Hourly imbalance shape:", imbalance_hourly.shape)
display(imbalance_native.head())
display(pd.DataFrame(imbalance_errors))


## 9. Energinet public wind forecast benchmark

This source is downloaded and retained, but it is **not yet labelled tradable**. The historical revision/publication structure must be verified before using it as a strict ex-ante benchmark.


In [ ]:
def prepare_energinet_wind_forecast(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()

    output = frame.copy()

    time_column = first_existing(
        output.columns,
        ["HourUTC", "TimeUTC", "Minutes5UTC", "QuarterUTC"],
    )
    area_column = first_existing(
        output.columns,
        ["PriceArea", "Area"],
    )

    required_columns = {
        "ForecastType",
        "ForecastDayAhead",
    }
    missing_columns = sorted(
        required_columns.difference(output.columns)
    )

    if missing_columns:
        raise KeyError(
            "Energinet forecast data are missing columns: "
            f"{missing_columns}"
        )

    output["timestamp_utc"] = pd.to_datetime(
        output[time_column],
        utc=True,
        errors="coerce",
    )
    output["price_area"] = (
        output[area_column].astype(str).str.strip()
    )
    output["forecast_type"] = (
        output["ForecastType"].astype(str).str.strip()
    )
    output["forecast_type_normalized"] = (
        output["forecast_type"]
        .str.lower()
        .str.replace(" ", "", regex=False)
        .str.replace("-", "", regex=False)
        .str.replace("_", "", regex=False)
    )
    output["forecast_day_ahead_mw"] = pd.to_numeric(
        output["ForecastDayAhead"],
        errors="coerce",
    )
    output = output.dropna(
        subset=[
            "timestamp_utc",
            "price_area",
            "forecast_day_ahead_mw",
        ]
    )

    exact_total_types = {
        "wind",
        "windpower",
        "totalwind",
        "totalwindpower",
    }
    exact_total_mask = output[
        "forecast_type_normalized"
    ].isin(exact_total_types)

    if exact_total_mask.any():
        wind_rows = output.loc[exact_total_mask].copy()
        aggregation_method = (
            "Energinet total-wind ForecastDayAhead"
        )
    else:
        component_mask = output[
            "forecast_type_normalized"
        ].str.contains("wind", case=False, na=False)
        wind_rows = output.loc[component_mask].copy()
        aggregation_method = (
            "Sum of Energinet wind-component "
            "ForecastDayAhead values"
        )

    if wind_rows.empty:
        available_types = sorted(
            output["forecast_type"]
            .dropna()
            .unique()
            .tolist()
        )
        raise ValueError(
            "No wind forecast rows were identified. "
            f"Available ForecastType values: {available_types}"
        )

    result = (
        wind_rows.groupby(
            ["timestamp_utc", "price_area"],
            as_index=False,
        )
        .agg(
            energinet_wind_forecast_mw=(
                "forecast_day_ahead_mw",
                "sum",
            ),
            energinet_forecast_types=(
                "forecast_type",
                lambda values: ",".join(
                    sorted(set(values))
                ),
            ),
        )
    )
    result["energinet_forecast_source_columns"] = (
        "ForecastDayAhead"
    )
    result["energinet_forecast_aggregation"] = (
        aggregation_method
    )
    result = trim_utc(result, "timestamp_utc")

    return (
        result.sort_values(
            ["timestamp_utc", "price_area"]
        )
        .drop_duplicates(
            ["timestamp_utc", "price_area"],
            keep="last",
        )
        .reset_index(drop=True)
    )


energinet_forecast_raw = pd.DataFrame()
energinet_wind_forecast = pd.DataFrame()
energinet_forecast_error = None

try:
    energinet_forecast_raw = eds_download_chunked(
        dataset="Forecasts_Hour",
        start=START_UTC - pd.Timedelta(days=1),
        end=END_UTC + pd.Timedelta(days=1),
        filters={"PriceArea": PRICE_AREAS},
        chunk_days=7 if RUN_MODE == "AUDIT" else 31,
    )

    print(
        "Available ForecastType values:",
        sorted(
            energinet_forecast_raw[
                "ForecastType"
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        ),
    )

    energinet_wind_forecast = (
        prepare_energinet_wind_forecast(
            energinet_forecast_raw
        )
    )

except Exception as exc:
    energinet_forecast_error = repr(exc)
    print(
        "Optional Energinet forecast benchmark failed:",
        energinet_forecast_error,
    )

print(
    "Energinet wind forecast shape:",
    energinet_wind_forecast.shape,
)
print(
    "Optional-source error:",
    energinet_forecast_error,
)
display(energinet_wind_forecast.head())


## 10. Main historical weather panel — Previous Runs D2

Each value is taken from the forecast at a fixed 48-hour lead. This is conservative relative to the 11:00 D-1 decision and produces a consistent information definition across the complete delivery day.


In [ ]:
OPEN_METEO_PREVIOUS_RUNS_URL = (
    "https://previous-runs-api.open-meteo.com/v1/forecast"
)


def calendar_date_chunks(
    start_date: str,
    end_date_exclusive: str,
    chunk_days: int,
) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    left = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date_exclusive).normalize()
    chunks: list[tuple[pd.Timestamp, pd.Timestamp]] = []

    while left < end:
        right = min(
            left + pd.Timedelta(days=chunk_days),
            end,
        )
        chunks.append((left, right))
        left = right

    return chunks


def parse_previous_runs_locations(
    payload: dict[str, Any] | list[Any],
    points: pd.DataFrame,
    model_alias: str,
    api_model: str,
) -> pd.DataFrame:
    items = payload if isinstance(payload, list) else [payload]

    if len(items) != len(points):
        raise DataDownloadError(
            "Previous Runs returned a different number of "
            f"locations: requested={len(points)}, "
            f"returned={len(items)}"
        )

    frames: list[pd.DataFrame] = []

    for (_, point), item in zip(
        points.reset_index(drop=True).iterrows(),
        items,
    ):
        hourly = item.get("hourly") if isinstance(item, dict) else None

        if not isinstance(hourly, dict) or "time" not in hourly:
            raise DataDownloadError(
                f"No hourly data for {model_alias} at "
                f"{point['point_id']}."
            )

        frame = pd.DataFrame(hourly)
        frame["timestamp_utc"] = pd.to_datetime(
            frame.pop("time"),
            utc=True,
            errors="coerce",
        )

        rename_map = {
            f"{variable}_previous_day{PREVIOUS_RUN_DAY}": variable
            for variable in WEATHER_VARIABLES
        }
        frame = frame.rename(columns=rename_map)

        for variable in WEATHER_VARIABLES:
            if variable not in frame.columns:
                frame[variable] = np.nan

        frame["model_alias"] = model_alias
        frame["api_model"] = api_model
        frame["lead_definition"] = (
            f"previous_day{PREVIOUS_RUN_DAY}"
        )
        frame["point_id"] = point["point_id"]
        frame["price_area"] = point["price_area"]
        frame["environment"] = point["environment"]
        frame["spatial_weight"] = point["spatial_weight"]
        frame["requested_latitude"] = point["latitude"]
        frame["requested_longitude"] = point["longitude"]
        frame["returned_latitude"] = item.get("latitude")
        frame["returned_longitude"] = item.get("longitude")
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)


def download_previous_runs_model(
    model_alias: str,
    api_model: str,
    start_date: str,
    end_date_exclusive: str,
    points: pd.DataFrame,
    chunk_days: int = 31,
    force_refresh: bool = False,
) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    chunks = calendar_date_chunks(
        start_date,
        end_date_exclusive,
        chunk_days,
    )

    for chunk_number, (left, right) in enumerate(
        chunks,
        start=1,
    ):
        api_end_inclusive = right - pd.Timedelta(days=1)

        params = {
            "latitude": ",".join(
                points["latitude"].astype(str)
            ),
            "longitude": ",".join(
                points["longitude"].astype(str)
            ),
            "hourly": ",".join(PREVIOUS_RUN_VARIABLES),
            "models": api_model,
            "start_date": left.date().isoformat(),
            "end_date": api_end_inclusive.date().isoformat(),
            "timezone": "UTC",
            "wind_speed_unit": "ms",
            "timeformat": "iso8601",
        }

        request_signature = stable_hash(
            {
                "model_alias": model_alias,
                "api_model": api_model,
                "start_date": params["start_date"],
                "end_date": params["end_date"],
                "variables": PREVIOUS_RUN_VARIABLES,
                "point_ids": points["point_id"].tolist(),
            }
        )

        cache_path = (
            RAW_DIR
            / "open_meteo_previous_runs"
            / model_alias
            / (
                f"{params['start_date']}_"
                f"{params['end_date']}_"
                f"{request_signature}.json"
            )
        )

        print(
            f"[previous-runs] {model_alias} "
            f"chunk {chunk_number}/{len(chunks)}: "
            f"{params['start_date']} to {params['end_date']}"
        )

        payload = request_json_cached(
            url=OPEN_METEO_PREVIOUS_RUNS_URL,
            params=params,
            cache_path=cache_path,
            force_refresh=force_refresh,
        )

        part = parse_previous_runs_locations(
            payload=payload,
            points=points,
            model_alias=model_alias,
            api_model=api_model,
        )
        parts.append(part)
        time.sleep(0.25)

    if not parts:
        return pd.DataFrame()

    output = pd.concat(parts, ignore_index=True)
    output = trim_utc(
        output,
        timestamp_column="timestamp_utc",
        start_utc=START_UTC,
        end_utc=END_UTC,
    )

    return (
        output.sort_values(
            [
                "timestamp_utc",
                "model_alias",
                "point_id",
            ]
        )
        .drop_duplicates(
            [
                "timestamp_utc",
                "model_alias",
                "point_id",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )


PREVIOUS_RUN_REQUEST_START_DATE = (
    START_UTC.floor("D").date().isoformat()
)
PREVIOUS_RUN_REQUEST_END_DATE_EXCLUSIVE = (
    END_UTC.ceil("D").date().isoformat()
)

print(
    "Exact modelling interval in UTC:",
    START_UTC,
    "to",
    END_UTC,
)
print(
    "Previous Runs request dates:",
    PREVIOUS_RUN_REQUEST_START_DATE,
    "to",
    PREVIOUS_RUN_REQUEST_END_DATE_EXCLUSIVE,
    "(exclusive)",
)

previous_runs_parts: list[pd.DataFrame] = []
previous_runs_errors: list[dict[str, Any]] = []

for model_alias, specification in MAIN_WEATHER_MODELS.items():
    try:
        part = download_previous_runs_model(
            model_alias=model_alias,
            api_model=specification["api_model"],
            start_date=PREVIOUS_RUN_REQUEST_START_DATE,
            end_date_exclusive=(
                PREVIOUS_RUN_REQUEST_END_DATE_EXCLUSIVE
            ),
            points=WEATHER_POINTS,
            chunk_days=7 if RUN_MODE == "AUDIT" else 31,
        )
        previous_runs_parts.append(part)
        print(
            f"{model_alias}: {len(part):,} point rows "
            "after exact UTC trimming"
        )
    except Exception as exc:
        previous_runs_errors.append(
            {
                "model_alias": model_alias,
                "api_model": specification["api_model"],
                "required": specification["required"],
                "error": repr(exc),
            }
        )
        print(f"{model_alias} failed:", repr(exc))

if previous_runs_parts:
    previous_runs_point = pd.concat(
        previous_runs_parts,
        ignore_index=True,
    )
    previous_runs_point = (
        previous_runs_point
        .sort_values(
            [
                "timestamp_utc",
                "model_alias",
                "point_id",
            ]
        )
        .drop_duplicates(
            [
                "timestamp_utc",
                "model_alias",
                "point_id",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )
else:
    previous_runs_point = pd.DataFrame()

previous_runs_error_report = pd.DataFrame(
    previous_runs_errors
)
previous_runs_error_report.to_csv(
    AUDIT_DIR / "previous_runs_download_errors.csv",
    index=False,
)

print(
    "Previous-runs point shape:",
    previous_runs_point.shape,
)
display(previous_runs_point.head())
display(previous_runs_error_report)


## 11. Aggregate D2 forecasts and select the stable source set


In [ ]:
def weighted_average(
    values: pd.Series,
    weights: pd.Series,
) -> float:
    valid = values.notna() & weights.notna()

    if not valid.any():
        return np.nan

    return float(
        np.average(
            values.loc[valid],
            weights=weights.loc[valid],
        )
    )


def aggregate_weather_by_area(
    point_forecasts: pd.DataFrame,
    extra_group_columns: list[str] | None = None,
) -> pd.DataFrame:
    if point_forecasts.empty:
        return pd.DataFrame()

    working = point_forecasts.copy()

    for variable in WEATHER_VARIABLES:
        working[variable] = pd.to_numeric(
            working[variable],
            errors="coerce",
        )

    for height in ["100m", "10m"]:
        radians = np.deg2rad(
            working[f"wind_direction_{height}"]
        )
        working[f"wind_direction_{height}_sin"] = np.sin(
            radians
        )
        working[f"wind_direction_{height}_cos"] = np.cos(
            radians
        )
        working[f"wind_speed_{height}_cubed"] = (
            working[f"wind_speed_{height}"].clip(lower=0) ** 3
        )

    value_columns = [
        "wind_speed_100m",
        "wind_speed_100m_cubed",
        "wind_direction_100m_sin",
        "wind_direction_100m_cos",
        "wind_speed_10m",
        "wind_speed_10m_cubed",
        "wind_direction_10m_sin",
        "wind_direction_10m_cos",
        "temperature_2m",
        "pressure_msl",
    ]

    group_columns = [
        "timestamp_utc",
        "price_area",
        "model_alias",
        "api_model",
    ]

    if extra_group_columns:
        group_columns.extend(extra_group_columns)

    rows: list[dict[str, Any]] = []

    for keys, group in working.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_columns, keys))

        masks = {
            "all": pd.Series(True, index=group.index),
            "land": group["environment"].isin(
                ["onshore", "coastal"]
            ),
            "offshore": group["environment"].eq("offshore"),
        }

        for region_name, mask in masks.items():
            subset = group.loc[mask]
            row[f"points_{region_name}"] = (
                subset["point_id"].nunique()
            )

            for column in value_columns:
                row[f"{column}_{region_name}"] = (
                    weighted_average(
                        subset[column],
                        subset["spatial_weight"],
                    )
                )

        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "timestamp_utc",
                "price_area",
                "model_alias",
            ]
        )
        .reset_index(drop=True)
    )


previous_runs_area_long = aggregate_weather_by_area(
    previous_runs_point,
    extra_group_columns=["lead_definition"],
)

if previous_runs_area_long.empty:
    raise RuntimeError(
        "No area-level Previous Runs weather data "
        "were produced."
    )

expected_area_hours = EXPECTED_HOURS * len(PRICE_AREAS)
coverage_rows: list[dict[str, Any]] = []
PRIMARY_WIND_HEIGHT_BY_MODEL: dict[str, str] = {}

for model_alias, specification in MAIN_WEATHER_MODELS.items():
    model_frame = previous_runs_area_long.loc[
        previous_runs_area_long["model_alias"].eq(model_alias)
    ]

    coverage_100m = (
        model_frame["wind_speed_100m_all"].notna().sum()
        / expected_area_hours
        if expected_area_hours
        else 0.0
    )
    coverage_10m = (
        model_frame["wind_speed_10m_all"].notna().sum()
        / expected_area_hours
        if expected_area_hours
        else 0.0
    )

    threshold = (
        MIN_CORE_SOURCE_COVERAGE
        if specification["required"]
        else MIN_OPTIONAL_SOURCE_COVERAGE
    )

    if coverage_100m >= threshold:
        selected_height = "100m"
        primary_coverage = coverage_100m
    elif coverage_10m >= threshold:
        selected_height = "10m"
        primary_coverage = coverage_10m
    else:
        selected_height = ""
        primary_coverage = max(
            coverage_100m,
            coverage_10m,
        )

    selected = bool(selected_height)

    if selected:
        PRIMARY_WIND_HEIGHT_BY_MODEL[
            model_alias
        ] = selected_height

    coverage_rows.append(
        {
            "model_alias": model_alias,
            "api_model": specification["api_model"],
            "provider": specification["provider"],
            "required": specification["required"],
            "coverage_wind_100m": coverage_100m,
            "coverage_wind_10m": coverage_10m,
            "selected_wind_height": selected_height,
            "primary_wind_coverage": primary_coverage,
            "selection_threshold": threshold,
            "selected": selected,
        }
    )

main_weather_coverage = pd.DataFrame(coverage_rows)

failed_core_sources = main_weather_coverage.loc[
    main_weather_coverage["required"]
    & ~main_weather_coverage["selected"]
]

display(main_weather_coverage)

if not failed_core_sources.empty:
    raise RuntimeError(
        "Required providers have no sufficiently complete "
        "historical wind channel: "
        f"{failed_core_sources['model_alias'].tolist()}"
    )

SELECTED_MAIN_SOURCES = (
    main_weather_coverage.loc[
        main_weather_coverage["selected"],
        "model_alias",
    ]
    .tolist()
)

if len(SELECTED_MAIN_SOURCES) < 3:
    raise RuntimeError(
        "Fewer than three stable historical wind providers "
        "were selected."
    )

standardized_parts: list[pd.DataFrame] = []

for model_alias in SELECTED_MAIN_SOURCES:
    selected_height = PRIMARY_WIND_HEIGHT_BY_MODEL[
        model_alias
    ]
    model_frame = previous_runs_area_long.loc[
        previous_runs_area_long["model_alias"].eq(
            model_alias
        )
    ].copy()

    for region_name in ["all", "land", "offshore"]:
        model_frame[
            f"wind_speed_primary_{region_name}"
        ] = model_frame[
            f"wind_speed_{selected_height}_{region_name}"
        ]
        model_frame[
            f"wind_speed_primary_cubed_{region_name}"
        ] = model_frame[
            f"wind_speed_{selected_height}_cubed_{region_name}"
        ]
        model_frame[
            f"wind_direction_primary_sin_{region_name}"
        ] = model_frame[
            f"wind_direction_{selected_height}_sin_{region_name}"
        ]
        model_frame[
            f"wind_direction_primary_cos_{region_name}"
        ] = model_frame[
            f"wind_direction_{selected_height}_cos_{region_name}"
        ]

    model_frame["selected_wind_height"] = selected_height
    standardized_parts.append(model_frame)

selected_previous_runs_area = pd.concat(
    standardized_parts,
    ignore_index=True,
)

valid_indices: dict[str, pd.MultiIndex] = {}

for model_alias in SELECTED_MAIN_SOURCES:
    valid_pairs = (
        selected_previous_runs_area.loc[
            selected_previous_runs_area[
                "model_alias"
            ].eq(model_alias)
            & selected_previous_runs_area[
                "wind_speed_primary_all"
            ].notna(),
            ["timestamp_utc", "price_area"],
        ]
        .drop_duplicates()
    )
    valid_indices[model_alias] = pd.MultiIndex.from_frame(
        valid_pairs
    )

complete_index = valid_indices[
    SELECTED_MAIN_SOURCES[0]
]

for model_alias in SELECTED_MAIN_SOURCES[1:]:
    complete_index = complete_index.intersection(
        valid_indices[model_alias]
    )

complete_pair_frame = complete_index.to_frame(
    index=False
)

selected_previous_runs_area = (
    selected_previous_runs_area.merge(
        complete_pair_frame,
        on=["timestamp_utc", "price_area"],
        how="inner",
        validate="many_to_one",
    )
    .sort_values(
        [
            "timestamp_utc",
            "price_area",
            "model_alias",
        ]
    )
    .reset_index(drop=True)
)

common_area_hours = len(complete_index)
common_panel_retention = (
    common_area_hours / expected_area_hours
    if expected_area_hours
    else 0.0
)

if common_panel_retention < MIN_COMMON_PANEL_RETENTION:
    raise RuntimeError(
        "The selected providers retain only "
        f"{common_panel_retention:.2%} of the requested panel."
    )

main_weather_coverage.to_csv(
    AUDIT_DIR / "main_weather_source_coverage.csv",
    index=False,
)

with (
    AUDIT_DIR / "selected_main_weather_sources.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        {
            "selected_sources": SELECTED_MAIN_SOURCES,
            "primary_wind_height_by_model": (
                PRIMARY_WIND_HEIGHT_BY_MODEL
            ),
            "main_start_date": MAIN_START_DATE,
            "main_end_date_exclusive": MAIN_END_DATE,
            "lead_definition": (
                f"previous_day{PREVIOUS_RUN_DAY}"
            ),
            "common_panel_retention": (
                common_panel_retention
            ),
        },
        file,
        indent=2,
    )

print("Selected main sources:", SELECTED_MAIN_SOURCES)
print(
    "Primary wind height by model:",
    PRIMARY_WIND_HEIGHT_BY_MODEL,
)
print("Common-panel retention:", common_panel_retention)
display(selected_previous_runs_area.head())


## 12. Build the main D2 modelling panel


In [ ]:
def weather_long_to_wide(
    weather_long: pd.DataFrame,
) -> pd.DataFrame:
    if weather_long.empty:
        return pd.DataFrame(
            columns=["timestamp_utc", "price_area"]
        )

    identifier_columns = {
        "timestamp_utc",
        "price_area",
        "model_alias",
        "api_model",
        "lead_definition",
        "run_init_utc",
        "assumed_available_utc",
        "decision_time_utc",
        "target_date_local",
        "information_time_safe",
    }

    feature_columns = [
        column
        for column in weather_long.columns
        if column not in identifier_columns
    ]

    wide_parts: list[pd.DataFrame] = []

    for feature in feature_columns:
        if not pd.api.types.is_numeric_dtype(
            weather_long[feature]
        ):
            continue

        pivot = weather_long.pivot_table(
            index=["timestamp_utc", "price_area"],
            columns="model_alias",
            values=feature,
            aggfunc="first",
        )
        pivot.columns = [
            f"{model_alias}__{feature}"
            for model_alias in pivot.columns
        ]
        wide_parts.append(pivot)

    return pd.concat(
        wide_parts,
        axis=1,
    ).reset_index()


main_weather_wide = weather_long_to_wide(
    selected_previous_runs_area
)

main_model_panel = actual_wind.merge(
    main_weather_wide,
    on=["timestamp_utc", "price_area"],
    how="inner",
    validate="one_to_one",
)

main_model_panel = main_model_panel.merge(
    day_ahead_prices[
        [
            "timestamp_utc",
            "price_area",
            "day_ahead_price_eur_mwh",
            "day_ahead_intervals_per_hour",
            "price_source_dataset",
        ]
    ],
    on=["timestamp_utc", "price_area"],
    how="left",
    validate="one_to_one",
)

if not imbalance_hourly.empty:
    main_model_panel = main_model_panel.merge(
        imbalance_hourly,
        on=["timestamp_utc", "price_area"],
        how="left",
        validate="one_to_one",
    )

if not energinet_wind_forecast.empty:
    main_model_panel = main_model_panel.merge(
        energinet_wind_forecast,
        on=["timestamp_utc", "price_area"],
        how="left",
        validate="one_to_one",
    )

main_model_panel = main_model_panel.sort_values(
    ["price_area", "timestamp_utc"]
).reset_index(drop=True)

local_timestamp = main_model_panel[
    "timestamp_utc"
].dt.tz_convert(LOCAL_TIMEZONE)

main_model_panel["target_date_local"] = (
    local_timestamp.dt.date.astype(str)
)
main_model_panel["hour_local"] = local_timestamp.dt.hour
main_model_panel["hour_utc"] = (
    main_model_panel["timestamp_utc"].dt.hour
)
main_model_panel["day_of_week"] = (
    local_timestamp.dt.dayofweek
)
main_model_panel["month"] = local_timestamp.dt.month
main_model_panel["is_weekend"] = (
    main_model_panel["day_of_week"].ge(5).astype(int)
)
main_model_panel["hour_local_sin"] = np.sin(
    2 * np.pi
    * main_model_panel["hour_local"]
    / 24
)
main_model_panel["hour_local_cos"] = np.cos(
    2 * np.pi
    * main_model_panel["hour_local"]
    / 24
)
main_model_panel["month_sin"] = np.sin(
    2 * np.pi * main_model_panel["month"] / 12
)
main_model_panel["month_cos"] = np.cos(
    2 * np.pi * main_model_panel["month"] / 12
)

if {
    "day_ahead_price_eur_mwh",
    "imbalance_price_eur_mwh",
}.issubset(main_model_panel.columns):
    main_model_panel["imbalance_spread_eur_mwh"] = (
        main_model_panel[
            "imbalance_price_eur_mwh"
        ]
        - main_model_panel[
            "day_ahead_price_eur_mwh"
        ]
    )

print("Main model panel shape:", main_model_panel.shape)
display(main_model_panel.head())


## 13. Exact Single Runs robustness panel

This panel uses the selected main source set, but only over the exact-run archive period. It is stored separately and is not concatenated with the D2 panel.


In [ ]:
def latest_eligible_run(
    decision_time_utc: pd.Timestamp,
    run_hours_utc: list[int],
    availability_lag_hours: int,
) -> pd.Timestamp:
    cutoff = (
        decision_time_utc
        - pd.Timedelta(hours=availability_lag_hours)
    )
    candidates: list[pd.Timestamp] = []

    for day_offset in range(0, 3):
        candidate_date = (
            cutoff.normalize()
            - pd.Timedelta(days=day_offset)
        )
        for run_hour in run_hours_utc:
            candidate = (
                candidate_date
                + pd.Timedelta(hours=run_hour)
            )
            if candidate <= cutoff:
                candidates.append(candidate)

    if not candidates:
        raise ValueError(
            f"No eligible run found before {decision_time_utc}."
        )

    return max(candidates)


def build_exact_run_manifest() -> pd.DataFrame:
    if not RUN_EXACT_ROBUSTNESS:
        return pd.DataFrame()

    rows: list[dict[str, Any]] = []

    target_dates = pd.date_range(
        EXACT_START_LOCAL.normalize(),
        EXACT_END_LOCAL.normalize(),
        freq="D",
        inclusive="left",
    )

    for target_start_local in target_dates:
        target_date_local = target_start_local.date()
        decision_date = target_date_local - timedelta(days=1)
        decision_time_local = pd.Timestamp(
            datetime.combine(
                decision_date,
                dt_time(
                    hour=DECISION_LOCAL_HOUR,
                    minute=0,
                ),
                tzinfo=LOCAL_TZ,
            )
        )
        decision_time_utc = (
            decision_time_local.tz_convert("UTC")
        )
        target_end_local = (
            target_start_local
            + pd.DateOffset(days=1)
        )

        for model_alias in SELECTED_MAIN_SOURCES:
            specification = EXACT_WEATHER_MODELS[
                model_alias
            ]
            run_init_utc = latest_eligible_run(
                decision_time_utc=decision_time_utc,
                run_hours_utc=specification[
                    "run_hours_utc"
                ],
                availability_lag_hours=specification[
                    "availability_lag_hours"
                ],
            )
            assumed_available_utc = (
                run_init_utc
                + pd.Timedelta(
                    hours=specification[
                        "availability_lag_hours"
                    ]
                )
            )

            rows.append(
                {
                    "target_date_local": (
                        target_date_local.isoformat()
                    ),
                    "model_alias": model_alias,
                    "api_model": specification[
                        "api_model"
                    ],
                    "decision_time_local": (
                        decision_time_local.isoformat()
                    ),
                    "decision_time_utc": (
                        decision_time_utc.isoformat()
                    ),
                    "run_init_utc": (
                        run_init_utc.isoformat()
                    ),
                    "assumed_available_utc": (
                        assumed_available_utc.isoformat()
                    ),
                    "availability_lag_hours": (
                        specification[
                            "availability_lag_hours"
                        ]
                    ),
                    "target_start_utc": (
                        target_start_local
                        .tz_convert("UTC")
                        .isoformat()
                    ),
                    "target_end_utc": (
                        target_end_local
                        .tz_convert("UTC")
                        .isoformat()
                    ),
                    "information_time_safe": (
                        assumed_available_utc
                        <= decision_time_utc
                    ),
                }
            )

    manifest = pd.DataFrame(rows)

    for column in [
        "decision_time_utc",
        "run_init_utc",
        "assumed_available_utc",
        "target_start_utc",
        "target_end_utc",
    ]:
        manifest[column] = pd.to_datetime(
            manifest[column],
            utc=True,
        )

    return manifest


exact_run_manifest = build_exact_run_manifest()

if not exact_run_manifest.empty:
    exact_run_manifest.to_csv(
        AUDIT_DIR / "exact_run_manifest.csv",
        index=False,
    )

display(exact_run_manifest.head())


In [ ]:
OPEN_METEO_SINGLE_RUN_URL = (
    "https://single-runs-api.open-meteo.com/v1/forecast"
)


def parse_single_run_locations(
    payload: dict[str, Any] | list[Any],
    points: pd.DataFrame,
    manifest_row: pd.Series,
) -> pd.DataFrame:
    items = payload if isinstance(payload, list) else [payload]

    if len(items) != len(points):
        raise DataDownloadError(
            "Single Runs returned a different number of "
            f"locations: requested={len(points)}, "
            f"returned={len(items)}"
        )

    frames: list[pd.DataFrame] = []

    for (_, point), item in zip(
        points.reset_index(drop=True).iterrows(),
        items,
    ):
        hourly = item.get("hourly") if isinstance(item, dict) else None

        if not isinstance(hourly, dict) or "time" not in hourly:
            raise DataDownloadError(
                f"No exact hourly data for "
                f"{manifest_row['model_alias']} at "
                f"{point['point_id']}."
            )

        frame = pd.DataFrame(hourly)
        frame["timestamp_utc"] = pd.to_datetime(
            frame.pop("time"),
            utc=True,
            errors="coerce",
        )
        frame["model_alias"] = manifest_row[
            "model_alias"
        ]
        frame["api_model"] = manifest_row[
            "api_model"
        ]
        frame["run_init_utc"] = manifest_row[
            "run_init_utc"
        ]
        frame["assumed_available_utc"] = (
            manifest_row["assumed_available_utc"]
        )
        frame["decision_time_utc"] = manifest_row[
            "decision_time_utc"
        ]
        frame["target_date_local"] = manifest_row[
            "target_date_local"
        ]
        frame["point_id"] = point["point_id"]
        frame["price_area"] = point["price_area"]
        frame["environment"] = point["environment"]
        frame["spatial_weight"] = point["spatial_weight"]
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)


def download_single_run(
    manifest_row: pd.Series,
    points: pd.DataFrame,
    force_refresh: bool = False,
) -> pd.DataFrame:
    run_init_utc = pd.Timestamp(
        manifest_row["run_init_utc"]
    )
    target_end_utc = pd.Timestamp(
        manifest_row["target_end_utc"]
    )

    forecast_hours = int(
        math.ceil(
            (
                target_end_utc - run_init_utc
            )
            / pd.Timedelta(hours=1)
        )
    ) + 2

    params = {
        "latitude": ",".join(
            points["latitude"].astype(str)
        ),
        "longitude": ",".join(
            points["longitude"].astype(str)
        ),
        "hourly": ",".join(WEATHER_VARIABLES),
        "models": manifest_row["api_model"],
        "run": run_init_utc.strftime(
            "%Y-%m-%dT%H:%M"
        ),
        "forecast_hours": forecast_hours,
        "timezone": "UTC",
        "wind_speed_unit": "ms",
        "timeformat": "iso8601",
    }

    cache_path = (
        RAW_DIR
        / "open_meteo_single_runs"
        / str(manifest_row["model_alias"])
        / (
            run_init_utc.strftime("%Y%m%dT%H%MZ")
            + ".json"
        )
    )

    payload = request_json_cached(
        url=OPEN_METEO_SINGLE_RUN_URL,
        params=params,
        cache_path=cache_path,
        force_refresh=force_refresh,
    )

    parsed = parse_single_run_locations(
        payload=payload,
        points=points,
        manifest_row=manifest_row,
    )

    target_start_utc = pd.Timestamp(
        manifest_row["target_start_utc"]
    )
    target_end_utc = pd.Timestamp(
        manifest_row["target_end_utc"]
    )

    return parsed.loc[
        parsed["timestamp_utc"].ge(target_start_utc)
        & parsed["timestamp_utc"].lt(target_end_utc)
    ].copy()


exact_parts: list[pd.DataFrame] = []
exact_errors: list[dict[str, Any]] = []

if RUN_EXACT_ROBUSTNESS:
    for row_number, manifest_row in (
        exact_run_manifest.iterrows()
    ):
        print(
            f"[exact] {row_number + 1}/"
            f"{len(exact_run_manifest)} "
            f"{manifest_row['model_alias']} "
            f"{manifest_row['target_date_local']}"
        )

        try:
            exact_parts.append(
                download_single_run(
                    manifest_row=manifest_row,
                    points=WEATHER_POINTS,
                )
            )
        except Exception as exc:
            exact_errors.append(
                {
                    "target_date_local": (
                        manifest_row[
                            "target_date_local"
                        ]
                    ),
                    "model_alias": manifest_row[
                        "model_alias"
                    ],
                    "api_model": manifest_row[
                        "api_model"
                    ],
                    "run_init_utc": manifest_row[
                        "run_init_utc"
                    ],
                    "error": repr(exc),
                }
            )

if exact_parts:
    exact_point_forecasts = pd.concat(
        exact_parts,
        ignore_index=True,
    )
else:
    exact_point_forecasts = pd.DataFrame()

exact_error_report = pd.DataFrame(exact_errors)
exact_error_report.to_csv(
    AUDIT_DIR / "exact_run_download_errors.csv",
    index=False,
)

if not exact_point_forecasts.empty:
    exact_area_long = aggregate_weather_by_area(
        exact_point_forecasts,
        extra_group_columns=[
            "run_init_utc",
            "assumed_available_utc",
            "decision_time_utc",
            "target_date_local",
        ],
    )
    exact_area_long["information_time_safe"] = (
        exact_area_long["assumed_available_utc"]
        <= exact_area_long["decision_time_utc"]
    )
    exact_weather_wide = weather_long_to_wide(
        exact_area_long
    )

    actual_exact = actual_wind.loc[
        actual_wind["timestamp_utc"].ge(
            EXACT_START_UTC
        )
        & actual_wind["timestamp_utc"].lt(
            EXACT_END_UTC
        )
    ].copy()

    exact_model_panel = actual_exact.merge(
        exact_weather_wide,
        on=["timestamp_utc", "price_area"],
        how="inner",
        validate="one_to_one",
    )
else:
    exact_area_long = pd.DataFrame()
    exact_weather_wide = pd.DataFrame()
    exact_model_panel = pd.DataFrame()

print("Exact point shape:", exact_point_forecasts.shape)
print("Exact area shape:", exact_area_long.shape)
print("Exact panel shape:", exact_model_panel.shape)
display(exact_error_report)


## 14. Data-quality and information-time audit


In [ ]:
def check_row(
    check: str,
    passed: bool,
    value: Any,
    threshold: Any,
    severity: str = "BLOCKING",
    notes: str = "",
) -> dict[str, Any]:
    return {
        "check": check,
        "status": "PASS" if passed else "FAIL",
        "value": value,
        "threshold": threshold,
        "severity": severity,
        "notes": notes,
    }


checks: list[dict[str, Any]] = []

checks.append(
    check_row(
        "actual wind row count",
        len(actual_wind) == EXPECTED_TARGET_ROWS,
        len(actual_wind),
        EXPECTED_TARGET_ROWS,
    )
)
checks.append(
    check_row(
        "actual wind duplicate rate",
        not actual_wind.duplicated(
            ["timestamp_utc", "price_area"]
        ).any(),
        float(
            actual_wind.duplicated(
                ["timestamp_utc", "price_area"]
            ).mean()
        ),
        0,
    )
)
checks.append(
    check_row(
        "day-ahead price coverage",
        day_ahead_prices[
            "day_ahead_price_eur_mwh"
        ].notna().mean()
        >= 0.99,
        float(
            day_ahead_prices[
                "day_ahead_price_eur_mwh"
            ].notna().mean()
        ),
        ">= 99%",
    )
)
checks.append(
    check_row(
        "required D2 weather sources",
        failed_core_sources.empty,
        (
            failed_core_sources[
                "model_alias"
            ].tolist()
        ),
        "no failed core source",
    )
)
checks.append(
    check_row(
        "selected weather source count",
        len(SELECTED_MAIN_SOURCES) >= 3,
        len(SELECTED_MAIN_SOURCES),
        ">= 4",
    )
)
checks.append(
    check_row(
        "homogeneous common-panel retention",
        (
            common_panel_retention
            >= MIN_COMMON_PANEL_RETENTION
        ),
        common_panel_retention,
        f">= {MIN_COMMON_PANEL_RETENTION:.1%}",
    )
)
checks.append(
    check_row(
        "main panel non-empty",
        not main_model_panel.empty,
        len(main_model_panel),
        "non-empty",
    )
)

if not imbalance_hourly.empty:
    checks.append(
        check_row(
            "imbalance price coverage",
            main_model_panel[
                "imbalance_price_eur_mwh"
            ].notna().mean()
            >= 0.95,
            float(
                main_model_panel[
                    "imbalance_price_eur_mwh"
                ].notna().mean()
            ),
            ">= 95%",
            severity="WARNING",
        )
    )

checks.append(
    check_row(
        "Energinet benchmark parsed",
        not energinet_wind_forecast.empty,
        len(energinet_wind_forecast),
        "non-empty",
        severity="WARNING",
        notes=(
            "Publication and revision timing remains "
            "a benchmark caveat."
        ),
    )
)

if RUN_EXACT_ROBUSTNESS:
    exact_safe_rate = (
        float(
            exact_run_manifest[
                "information_time_safe"
            ].mean()
        )
        if not exact_run_manifest.empty
        else 0.0
    )
    checks.append(
        check_row(
            "exact-run information-time safety",
            exact_safe_rate == 1.0,
            exact_safe_rate,
            1.0,
        )
    )
    checks.append(
        check_row(
            "exact robustness source count",
            (
                exact_area_long["model_alias"].nunique()
                >= 3
                if not exact_area_long.empty
                else False
            ),
            (
                exact_area_long["model_alias"].nunique()
                if not exact_area_long.empty
                else 0
            ),
            ">= 4",
            severity="WARNING",
        )
    )

quality_report = pd.DataFrame(checks)

blocking_failures = quality_report.loc[
    quality_report["status"].eq("FAIL")
    & quality_report["severity"].eq("BLOCKING")
]

display(quality_report)
print("Blocking failures:", len(blocking_failures))


## 15. Visual sanity checks


In [ ]:
for price_area in PRICE_AREAS:
    area_frame = actual_wind.loc[
        actual_wind["price_area"].eq(price_area)
    ]

    figure, axis = plt.subplots(figsize=(14, 4))
    axis.plot(
        area_frame["timestamp_utc"],
        area_frame["actual_wind_mwh"],
        label="Actual wind",
    )

    if not energinet_wind_forecast.empty:
        benchmark = energinet_wind_forecast.loc[
            energinet_wind_forecast[
                "price_area"
            ].eq(price_area)
        ]
        axis.plot(
            benchmark["timestamp_utc"],
            benchmark[
                "energinet_wind_forecast_mw"
            ],
            label="Energinet forecast",
            alpha=0.8,
        )

    axis.set_title(
        f"{price_area}: actual wind and public benchmark"
    )
    axis.set_ylabel("MW / MWh per hour")
    axis.set_xlabel("UTC")
    axis.legend()
    figure.tight_layout()
    figure.savefig(
        FIGURE_DIR
        / f"notebook_01_{price_area.lower()}_wind.png",
        dpi=150,
    )
    plt.show()


for price_area in PRICE_AREAS:
    plot_frame = (
        selected_previous_runs_area.loc[
            selected_previous_runs_area[
                "price_area"
            ].eq(price_area)
        ]
        .pivot_table(
            index="timestamp_utc",
            columns="model_alias",
            values="wind_speed_100m_all",
            aggfunc="first",
        )
        .sort_index()
    )

    axis = plot_frame.plot(
        figsize=(14, 5),
        title=(
            f"{price_area}: D2 100 m wind-speed "
            "forecasts by provider"
        ),
    )
    axis.set_ylabel("m/s")
    axis.set_xlabel("UTC")
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR
        / (
            f"notebook_01_"
            f"{price_area.lower()}_d2_weather.png"
        ),
        dpi=150,
    )
    plt.show()


## 16. Save validated outputs and verification report


In [ ]:
OUTPUT_FILES = {
    "actual_wind": (
        INTERIM_DIR / "actual_wind_hourly.parquet"
    ),
    "day_ahead_prices": (
        INTERIM_DIR / "day_ahead_prices_hourly.parquet"
    ),
    "imbalance_native": (
        INTERIM_DIR / "imbalance_prices_native.parquet"
    ),
    "imbalance_hourly": (
        INTERIM_DIR / "imbalance_prices_hourly.parquet"
    ),
    "energinet_forecast": (
        INTERIM_DIR
        / "energinet_wind_forecast_hourly.parquet"
    ),
    "previous_runs_point": (
        INTERIM_DIR
        / "weather_previous_runs_d2_point.parquet"
    ),
    "previous_runs_area": (
        INTERIM_DIR
        / "weather_previous_runs_d2_area_long.parquet"
    ),
    "main_model_panel": (
        PROCESSED_DIR
        / "wind_model_panel_d2_hourly.parquet"
    ),
    "exact_point": (
        INTERIM_DIR
        / "weather_exact_runs_point.parquet"
    ),
    "exact_area": (
        INTERIM_DIR
        / "weather_exact_runs_area_long.parquet"
    ),
    "exact_model_panel": (
        PROCESSED_DIR
        / "wind_model_panel_exact_runs_hourly.parquet"
    ),
}

actual_wind.to_parquet(
    OUTPUT_FILES["actual_wind"],
    index=False,
)
day_ahead_prices.to_parquet(
    OUTPUT_FILES["day_ahead_prices"],
    index=False,
)

if not imbalance_native.empty:
    imbalance_native.to_parquet(
        OUTPUT_FILES["imbalance_native"],
        index=False,
    )

if not imbalance_hourly.empty:
    imbalance_hourly.to_parquet(
        OUTPUT_FILES["imbalance_hourly"],
        index=False,
    )

if not energinet_wind_forecast.empty:
    energinet_wind_forecast.to_parquet(
        OUTPUT_FILES["energinet_forecast"],
        index=False,
    )

previous_runs_point.to_parquet(
    OUTPUT_FILES["previous_runs_point"],
    index=False,
)
selected_previous_runs_area.to_parquet(
    OUTPUT_FILES["previous_runs_area"],
    index=False,
)
main_model_panel.to_parquet(
    OUTPUT_FILES["main_model_panel"],
    index=False,
)

if not exact_point_forecasts.empty:
    exact_point_forecasts.to_parquet(
        OUTPUT_FILES["exact_point"],
        index=False,
    )
if not exact_area_long.empty:
    exact_area_long.to_parquet(
        OUTPUT_FILES["exact_area"],
        index=False,
    )
if not exact_model_panel.empty:
    exact_model_panel.to_parquet(
        OUTPUT_FILES["exact_model_panel"],
        index=False,
    )

quality_report.to_csv(
    AUDIT_DIR / "notebook_01_quality_report.csv",
    index=False,
)

verification_status = (
    "READY_FOR_FULL_DOWNLOAD"
    if blocking_failures.empty
    and RUN_MODE == "AUDIT"
    else "READY_FOR_MODELLING"
    if blocking_failures.empty
    and RUN_MODE == "FULL"
    else "NOT_READY"
)

verification_report = {
    "verification_status": verification_status,
    "run_mode": RUN_MODE,
    "main_start_local": START_LOCAL.isoformat(),
    "main_end_local_exclusive": END_LOCAL.isoformat(),
    "main_lead_definition": (
        f"previous_day{PREVIOUS_RUN_DAY}"
    ),
    "selected_main_sources": SELECTED_MAIN_SOURCES,
    "source_coverage": (
        main_weather_coverage.to_dict(
            orient="records"
        )
    ),
    "common_panel_retention": (
        common_panel_retention
    ),
    "actual_wind_rows": len(actual_wind),
    "day_ahead_rows": len(day_ahead_prices),
    "imbalance_native_rows": len(
        imbalance_native
    ),
    "energinet_forecast_rows": len(
        energinet_wind_forecast
    ),
    "previous_runs_point_rows": len(
        previous_runs_point
    ),
    "previous_runs_area_rows": len(
        selected_previous_runs_area
    ),
    "main_model_panel_shape": list(
        main_model_panel.shape
    ),
    "exact_start_local": (
        EXACT_START_LOCAL.isoformat()
        if RUN_EXACT_ROBUSTNESS
        else None
    ),
    "exact_end_local_exclusive": (
        EXACT_END_LOCAL.isoformat()
        if RUN_EXACT_ROBUSTNESS
        else None
    ),
    "exact_weather_point_rows": len(
        exact_point_forecasts
    ),
    "exact_weather_area_rows": len(
        exact_area_long
    ),
    "exact_model_panel_shape": list(
        exact_model_panel.shape
    ),
    "previous_runs_errors": (
        previous_runs_errors
    ),
    "exact_run_errors": exact_errors,
    "energinet_forecast_error": (
        energinet_forecast_error
    ),
    "blocking_failures": (
        blocking_failures.to_dict(
            orient="records"
        )
    ),
    "output_files": {
        key: {
            "path": str(path),
            "exists": path.exists(),
            "size_mb": (
                round(
                    path.stat().st_size
                    / (1024 ** 2),
                    3,
                )
                if path.exists()
                else None
            ),
        }
        for key, path in OUTPUT_FILES.items()
    },
}

with (
    AUDIT_DIR
    / "notebook_01_verification.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        verification_report,
        file,
        indent=2,
        default=str,
    )

print("=== NOTEBOOK 01 VERIFICATION ===")
print(
    json.dumps(
        verification_report,
        indent=2,
        default=str,
    )
)
print("=== END VERIFICATION ===")


## Acceptance gate

Proceed to Notebook 02 only when:

- the audit run returns `READY_FOR_FULL_DOWNLOAD`;
- the full run returns `READY_FOR_MODELLING`;
- all three mandatory long-history providers pass wind coverage;
- any additional provider is selected only from observed wind-variable coverage;
- the selected source set remains homogeneous over the full main period;
- exact runs remain a separate robustness panel.

Notebook 02 will train source-specific LightGBM experts for every selected provider and compare individual experts, equal weighting, constrained static weighting, dynamic weighting, stacking and pooled LightGBM.
